# NegotiateEnv Training on Google Colab Pro

This notebook trains an LLM agent to negotiate B2B SaaS contracts using the NegotiateEnv OpenEnv environment.

**Architecture:**
- **Environment Server**: Deployed on HuggingFace Spaces (always accessible)
- **Training**: Runs here in Colab Pro (connects to HF Spaces)

**GPU Requirements:**
- **Colab Pro (A100/V100)**: TRL GRPO training (~1 hour) ✅ RECOMMENDED
- **Free Colab (T4)**: Unsloth 4-bit LoRA training (~2-3 hours)

**What this notebook does:**
1. Connects to your HuggingFace Spaces environment
2. Installs training dependencies
3. Runs baseline evaluation
4. Trains the model (TRL GRPO or Unsloth)
5. Evaluates trained model
6. Plots reward curves and strategy distribution
7. Saves model to HuggingFace Hub

## 1. Configuration: Set Your HuggingFace Spaces URL

In [ ]:
# ⚠️ IMPORTANT: Replace with your actual HuggingFace Spaces URL
# Your URL after deployment:
# https://kushaladhyaru-negotiate-env.hf.space

ENV_URL = "https://kushaladhyaru-negotiate-env.hf.space"

print(f"Environment URL: {ENV_URL}")
print("\n⚠️  Make sure you've deployed the environment to HuggingFace Spaces first!")
print("   See DEPLOY_INSTRUCTIONS.md for step-by-step guide.")

## 2. Setup: Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/kushal511/negotiate-env.git
%cd negotiate-env

In [ ]:
# Check GPU type
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# Install base dependencies (no server needed - we use HF Spaces)
!pip install -e .
!pip install requests matplotlib numpy

## 3. Test Connection to HuggingFace Spaces Environment

In [ ]:
# Test connection to your deployed environment
import requests
import json

print(f"Testing connection to: {ENV_URL}")

try:
    # Test health endpoint
    response = requests.get(f"{ENV_URL}/health", timeout=10)
    if response.status_code == 200:
        print("✅ Environment server is accessible!")
        print(f"   Health check: {response.json()}")
    else:
        print(f"❌ Server returned status code: {response.status_code}")
        print(f"   Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Failed to connect to environment server")
    print(f"   Error: {e}")
    print("\n⚠️  Make sure:")
    print("   1. You've deployed to HuggingFace Spaces")
    print("   2. Your Space is running (check the Space page)")
    print("   3. ENV_URL is correct (should end with .hf.space)")
    raise

# Test reset endpoint
try:
    response = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
    if response.status_code == 200:
        obs = response.json()
        print(f"\n✅ Environment reset successful!")
        print(f"   Scenario: {obs.get('context', '')[:100]}...")
        print(f"   Max turns: {obs.get('max_turns')}")
    else:
        print(f"\n⚠️  Reset returned status code: {response.status_code}")
except Exception as e:
    print(f"\n⚠️  Reset test failed: {e}")

## 4. Run Baseline Evaluation (Before Training)

In [ ]:
# Evaluate random baseline
!python baseline_random.py --episodes 50 --env-url {ENV_URL}

In [ ]:
# Evaluate rule-based baseline
!python baseline_rule.py --episodes 50 --env-url {ENV_URL}

In [ ]:
# Full evaluation with metrics
!python evaluate.py --agent rule --episodes 100 --env-url {ENV_URL}

## 5. Choose Training Method Based on GPU

In [ ]:
# Detect GPU and recommend training method
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()

print(f"Detected GPU: {gpu_name}")

if "A100" in gpu_name or "V100" in gpu_name:
    print("\n✅ Recommended: TRL GRPO Training (faster, better performance)")
    print("   Run the 'Option A: TRL GRPO Training' cell below")
elif "T4" in gpu_name:
    print("\n✅ Recommended: Unsloth 4-bit LoRA Training (memory efficient)")
    print("   Run the 'Option B: Unsloth Training' cell below")
else:
    print(f"\n⚠️  Unknown GPU: {gpu_name}")
    print("   Try Unsloth training first (more memory efficient)")

## Option A: TRL GRPO Training (A100/V100)

Use this for Colab Pro with A100 or V100 GPUs.

In [ ]:
# Install TRL training dependencies
!pip install trl>=0.29.0 transformers>=4.50.0 accelerate>=0.34.0 peft>=0.13.0
!pip install vllm>=0.6.0 torch>=2.5.0

In [ ]:
# Run TRL GRPO training (colocate mode - single GPU)
# Connects to your HuggingFace Spaces environment
!python train_negotiate.py \
    --vllm-mode colocate \
    --model-name Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-grpo-output \
    --num-train-epochs 3 \
    --per-device-train-batch-size 4 \
    --learning-rate 5e-7 \
    --env-url {ENV_URL}

## Option B: Unsloth 4-bit LoRA Training (T4 or any GPU)

Use this for free Colab T4 or if you have memory constraints.

In [ ]:
# Install Unsloth (optimized for Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl>=0.29.0 datasets requests

In [ ]:
# Run Unsloth training
# Connects to your HuggingFace Spaces environment
!python train_negotiate_unsloth.py \
    --env-url {ENV_URL} \
    --model-name Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-unsloth-output \
    --num-episodes 200 \
    --learning-rate 5e-5 \
    --per-device-train-batch-size 2

## 5. Plot Training Results

In [ ]:
# Install plotting dependencies
!pip install matplotlib numpy

In [ ]:
# Plot reward curve
import os

# Determine which output directory exists
if os.path.exists("negotiate-grpo-output/trainer_state.json"):
    !python plot_reward_curve.py --log-file negotiate-grpo-output/trainer_state.json --out reward_curve.png
elif os.path.exists("negotiate-unsloth-output/trainer_state.json"):
    !python plot_reward_curve.py --log-file negotiate-unsloth-output/trainer_state.json --out reward_curve.png
else:
    print("⚠️  No training log found. Make sure training completed successfully.")

In [ ]:
# Display reward curve
from IPython.display import Image, display

if os.path.exists("reward_curve.png"):
    display(Image("reward_curve.png"))
else:
    print("⚠️  Reward curve not generated")

## 6. Evaluate Trained Model

Note: This requires OpenAI API key for LLM evaluation. Skip if you don't have one.

In [ ]:
# Set your OpenAI API key (optional - only for LLM evaluation)
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key

In [ ]:
# Evaluate trained model (if you have OpenAI API key)
# !python evaluate.py --agent llm --model gpt-4o-mini --episodes 50

## 7. Run Demo with Trained Model

In [ ]:
# Run demo with rule-based agent
!python demo.py --difficulty medium

## 8. Save Model to HuggingFace Hub

In [ ]:
# Install huggingface_hub
!pip install huggingface_hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Push model to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()

# Determine which output directory to upload
if os.path.exists("negotiate-grpo-output"):
    output_dir = "negotiate-grpo-output"
elif os.path.exists("negotiate-unsloth-output"):
    output_dir = "negotiate-unsloth-output"
else:
    raise ValueError("No trained model found")

# Your HuggingFace username
repo_id = "KushalAdhyaru/negotiate-env-qwen-1.5b"

print(f"Uploading {output_dir} to {repo_id}...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_id,
    repo_type="model",
)
print(f"✅ Model uploaded to https://huggingface.co/{repo_id}")

## 9. Download Results for Submission

In [ ]:
# Create a zip file with all results
!zip -r training_results.zip \
    reward_curve.png \
    negotiate-*-output/ \
    *.log

print("\n✅ Results saved to training_results.zip")
print("Download this file from Colab's file browser (left sidebar)")

## Summary

**What you've accomplished:**
1. ✅ Set up NegotiateEnv environment server
2. ✅ Ran baseline evaluations (random + rule-based)
3. ✅ Trained LLM agent using GRPO or Unsloth
4. ✅ Generated reward curve visualization
5. ✅ Saved trained model to HuggingFace Hub

**Next steps for hackathon submission:**
1. Deploy environment to HuggingFace Spaces
2. Make dataset public on HuggingFace
3. Push code to public GitHub repo
4. Create demo video/screenshots
5. Submit to hackathon portal